# 02 — Evaluation: ROUGE-L · BERTScore · AlignScore

This notebook evaluates the fine-tuned Mistral-7B summarizer against the test set using three complementary metrics:

| Metric | What it measures |
|--------|------------------|
| **ROUGE-L** | Longest Common Subsequence overlap (lexical) |
| **BERTScore** | Semantic similarity via DeBERTa-xlarge embeddings |
| **AlignScore** | Factual consistency — does the summary align with the source? |

**Prerequisites:** Run `01_training.ipynb` first to produce the adapter and predictions.

In [ ]:
import subprocess, sys
!pip install -q rouge-score bert-score PyYAML
# AlignScore install
!pip install -q git+https://github.com/yuh-zha/AlignScore.git
!python -m spacy download en_core_web_sm -q
print('✅ Metric libraries installed')

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open('config/eval_config.yaml') as f:
    config = yaml.safe_load(f)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Config loaded:', json.dumps(config['metrics'], indent=2))

## Step 1: Generate Predictions (if not already done)

In [ ]:
predictions_path = config['output']['predictions_file']

if not os.path.isfile(predictions_path):
    print('Predictions file not found — running inference ...')
    from src.evaluation.generate_summaries import run_inference
    results = run_inference(config)
    print(f'Generated {len(results)} predictions')
else:
    print(f'Using existing predictions: {predictions_path}')
    with open(predictions_path) as f:
        results = [json.loads(l) for l in f]
    print(f'Loaded {len(results)} examples')

In [ ]:
sources     = [r['source']     for r in results]
references  = [r['reference']  for r in results]
predictions = [r['prediction'] for r in results]

print('=== Sample ===')
print('Source    :', sources[0][:300])
print('Reference :', references[0])
print('Prediction:', predictions[0])

## Step 2: ROUGE Scores

In [ ]:
from src.evaluation.evaluate import compute_rouge

rouge_scores = compute_rouge(predictions, references)

print('ROUGE Scores:')
for k, v in rouge_scores.items():
    print(f'  {k:<12}: {v:.4f}')

## Step 3: BERTScore

In [ ]:
from src.evaluation.evaluate import compute_bertscore

bsc = config['metrics']['bertscore']
bertscore_scores = compute_bertscore(
    predictions, references,
    model_type=bsc['model_type'],
    batch_size=bsc['batch_size'],
    device=bsc.get('device', 'cuda'),
)

print('BERTScore:')
for k, v in bertscore_scores.items():
    print(f'  {k:<30}: {v:.4f}')

## Step 4: AlignScore

In [ ]:
from src.evaluation.evaluate import compute_alignscore

asc = config['metrics']['alignscore']
alignscore_result = compute_alignscore(
    predictions=predictions,
    sources=sources,
    ckpt_path=asc['ckpt_path'],
    evaluation_mode=asc['evaluation_mode'],
    batch_size=asc['batch_size'],
    device=asc.get('device', 'cuda'),
)

print('AlignScore:', alignscore_result)

## Step 5: Results Summary & Visualization

In [ ]:
all_results = {
    'num_examples': len(predictions),
    **rouge_scores,
    **bertscore_scores,
    **alignscore_result,
}

# Save
os.makedirs('results', exist_ok=True)
with open(config['output']['metrics_file'], 'w') as f:
    json.dump(all_results, f, indent=2)
print('Results saved to', config['output']['metrics_file'])

# Display table
df = pd.DataFrame([{'Metric': k, 'Score': v} for k, v in all_results.items() if isinstance(v, float)])
print(df.to_string(index=False))

In [ ]:
# Bar chart of all metrics
metric_map = {
    'rouge1': 'ROUGE-1', 'rouge2': 'ROUGE-2', 'rougeL': 'ROUGE-L', 'rougeLsum': 'ROUGE-Lsum',
    'bertscore_precision': 'BERTScore-P', 'bertscore_recall': 'BERTScore-R', 'bertscore_f1': 'BERTScore-F1',
    'alignscore': 'AlignScore',
}

labels, values, colors = [], [], []
color_groups = {
    'rouge': '#6366f1', 'bertscore': '#10b981', 'align': '#f59e0b'
}

for k, label in metric_map.items():
    if k in all_results and all_results[k] is not None:
        labels.append(label)
        values.append(all_results[k])
        if 'rouge' in k: colors.append(color_groups['rouge'])
        elif 'bertscore' in k: colors.append(color_groups['bertscore'])
        else: colors.append(color_groups['align'])

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, min(1.0, max(values) * 1.15))
ax.set_title('Evaluation Metrics — Mistral-7B QLoRA Research Summarizer', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.tick_params(axis='x', rotation=30)

from matplotlib.patches import Patch
legend = [Patch(color=color_groups['rouge'], label='ROUGE'),
          Patch(color=color_groups['bertscore'], label='BERTScore'),
          Patch(color=color_groups['align'], label='AlignScore')]
ax.legend(handles=legend, loc='upper right')

plt.tight_layout()
plt.savefig('results/eval_metrics_chart.png', bbox_inches='tight')
plt.show()
print('Chart saved to results/eval_metrics_chart.png')

## Step 6: Per-Example Analysis

In [ ]:
# Per-example ROUGE-L
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

per_rougeL = [scorer.score(ref, pred)['rougeL'].fmeasure
              for ref, pred in zip(references, predictions)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(per_rougeL, bins=40, color='#6366f1', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(per_rougeL), color='#ef4444', linestyle='--', label=f'Mean: {np.mean(per_rougeL):.3f}')
ax.set_xlabel('ROUGE-L F1')
ax.set_ylabel('Count')
ax.set_title('Per-example ROUGE-L Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print(f'ROUGE-L — Mean: {np.mean(per_rougeL):.4f} | Median: {np.median(per_rougeL):.4f}')
print(f'           P25: {np.percentile(per_rougeL, 25):.4f} | P75: {np.percentile(per_rougeL, 75):.4f}')

In [ ]:
# Show top-5 best and worst examples by ROUGE-L
sorted_idx = np.argsort(per_rougeL)

print('=== WORST 3 EXAMPLES (lowest ROUGE-L) ===')
for i in sorted_idx[:3]:
    print(f'[ROUGE-L={per_rougeL[i]:.3f}]')
    print(f'  Reference : {references[i][:200]}')
    print(f'  Prediction: {predictions[i][:200]}\n')

print('=== BEST 3 EXAMPLES (highest ROUGE-L) ===')
for i in sorted_idx[-3:]:
    print(f'[ROUGE-L={per_rougeL[i]:.3f}]')
    print(f'  Reference : {references[i][:200]}')
    print(f'  Prediction: {predictions[i][:200]}\n')